# Analysing Damodaran's S&P500 analysis

AIM: check if Damodaran's analysis suggest it is an appropriate time to invest in the S&P500

Aswart Damodaran ([website](https://pages.stern.nyu.edu/~adamodar/New_Home_Page/home.htm)) is a SternU professor.
I watched most of his teeaching materials, including his full valuation class.
He is very analytical in the way he dissects the components that contribute to the value of a company.
Similarly, and this is relevant to this notebook, he decomposes the value of stock expected returns, using the S&P proxy.

Basically, the expected stock return is decomposed into:
- RFR (Risk Free Rate), proxied as the 10Y US Treasury yield that is causally related to the expected USD inflation and therefore also the future stability of United States
- Implied ERP (Equity Risk Premium), which is computed from the current S&P500 price and the expected future earnings. It captures the willingnes of investors to risk over equities to have higher return at the price of possible drawdowns.

He updates this analysis every first day of the month.

Note that this analysis is US-centric, but considering that US is currently 55% of the world market cap (FTSE All World Index, May 2023), and geopolitically the driver of the open-market economy, it is fair to focus on it, also considering the larger availability of data.


In [1]:
import pandas as pd # Note: needs xlrd and openpyxl to be installed to read excel files
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ssl
from datetime import datetime

pd.set_option('plotting.backend', 'plotly')
pio.renderers.default = 'notebook_connected'
ssl._create_default_https_context = ssl._create_unverified_context # Needed to download data

In [2]:
# There are two source of data:
# - yearly data, since 1960
# - monthly data, since September 2008
url_yearly = "https://pages.stern.nyu.edu/~adamodar/pc/datasets/histimpl.xls"
url_monthly = "https://pages.stern.nyu.edu/~adamodar/pc/implprem/ERPbymonth.xlsx"
url_dayly_2008 = "https://pages.stern.nyu.edu/~adamodar/pc/blog/ERPbyDay2008Crisis.xlsx"
url_dayly_covid = "https://pages.stern.nyu.edu/~adamodar/pc/blog/ERPbyDayCOVID.xlsx"
url_dayly_tariff = "https://pages.stern.nyu.edu/~adamodar/pc/blog/TariffERPbyday.xlsx"


LAST_YEAR = datetime.now().year-1 # Modify accordingly, to skip the part under the main table
dfy = (
    pd.read_excel(url_yearly, sheet_name="Historical Impl Premiums", skiprows=6, nrows=LAST_YEAR-1959)
)
dfm = (
    pd.read_excel(url_monthly, sheet_name="Historical ERP")
)
dfd = {
    "2008": pd.read_excel(url_dayly_2008, sheet_name="Main Data", skiprows=2),
    "COVID": pd.read_excel(url_dayly_covid, sheet_name="Raw Data", skiprows=0),
    "TARIFF": pd.read_excel(url_dayly_tariff, sheet_name="Sheet1", skiprows=0)
}

/Users/danieleongari/opt/anaconda3/envs/py310/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning:

Unknown extension is not supported and will be removed

/Users/danieleongari/opt/anaconda3/envs/py310/lib/python3.10/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning:

Cannot parse header or footer so it will be ignored



In [3]:
# Checkpoint to reload original data if I mess up
dfm_orig, dfy_orig, dfd_orig = dfm.copy(), dfy.copy(), dfd.copy()

In [4]:
# Clean data and make column names consistent
dfy, dfm, dfd = dfy_orig.copy(), dfm_orig.copy(), dfd_orig.copy()

# Identify for all: rERP, T.Bond Rate, S&P 500, rGROWTH (where r stand for reference)
dfy = (
    dfy
    .assign(
        Date=lambda x: pd.to_datetime((x["Year"]+1).astype(str)), # Date refers to the end of the year, so I will add 1 to convert it to the first day of the next year
        rERP=lambda x: x["Implied ERP (FCFE)"],
        rGROWTH=lambda x: x["Analyst Growth Estimate"],
        ) 
)
display(dfy)
dfm = (
    dfm
    .assign(
        Date=lambda x: pd.to_datetime(x["Start of month"]),
        rERP=lambda x: x["ERP (T12m)"],
        rGROWTH=lambda x: x["Expected growth rate"],
        ) 
    .dropna(subset=["Date"])
    .drop(index=192) # problemetic data
)
display(dfm)
dfd = (
    pd.concat([
    (
        dfd["2008"]
        .assign(
            Date=lambda x: pd.to_datetime(x["Date"]),
            rERP=lambda x: x["Implied Premium"],
            rGROWTH=lambda x: x["Earnings g- next 5 years"],
        )
        .assign(**{"T.Bond Rate": lambda x: x["10-yr Treasuries"]})
    ),
    (
        dfd["COVID"]
        .rename(columns={"ERP":"rERP"})
        .assign(rGROWTH=pd.NA)
        .assign(Date=lambda x: pd.to_datetime(x["Date"]))
        .assign(**{"T.Bond Rate": lambda x: x["T. Bond Rate"]})
    ),
        dfd["TARIFF"]
        .assign(
            Date=lambda x: pd.to_datetime(x["Date"]),
            rERP=lambda x: x["Implied ERP"],
            rGROWTH=pd.NA
        )
        .assign(**{"T.Bond Rate": lambda x: x["T. Bond Rate"]})
        .dropna(subset=["rERP"])
    ])
    # Now fill the dates in between to avoid straight lines in the plot
    .set_index("Date")
    .pipe(
        lambda df: df.reindex(
            pd.date_range(
                start=df.index.min(),
                end=df.index.max(),
                freq="D"
            )
        )
    )
    .reset_index()
    .rename(columns={"index": "Date"})
)
display(dfd[["Date", "rERP", "T.Bond Rate", "rGROWTH", "S&P 500"]])

,Year,Earnings Yield,Dividend Yield,S&P 500,Earnings*,Dividends*,Dividends + Buybacks,Change in Earnings,Change in Dividends,T.Bill Rate,...,Bond-Bill,Smoothed Growth,Implied Premium (DDM),Analyst Growth Estimate,Implied ERP (FCFE),Implied Premium (FCFE with sustainable Payout),ERP/Riskfree Rate,Date,rERP,rGROWTH
0,1960,0.053400,0.034100,58.11,3.103074,1.981551,NaN,NaN,NaN,0.0266,...,0.0010,0.024484,NaN,NaN,NaN,NaN,NaN,1961-01-01,NaN,NaN
1,1961,0.047100,0.028500,71.55,3.370005,2.039175,NaN,0.086021,0.029080,0.0213,...,0.0022,0.024051,0.0292,NaN,0.0292,NaN,1.242553,1962-01-01,0.0292,NaN
2,1962,0.058100,0.034000,63.10,3.666110,2.145400,NaN,0.087865,0.052092,0.0273,...,0.0112,0.040496,0.0356,NaN,0.0356,NaN,0.924675,1963-01-01,0.0356,NaN
3,1963,0.055100,0.031300,75.02,4.133602,2.348126,NaN,0.127517,0.094493,0.0312,...,0.0102,0.049635,0.0338,NaN,0.0338,NaN,0.816425,1964-01-01,0.0338,NaN
4,1964,0.056200,0.030500,84.75,4.762950,2.584875,NaN,0.152252,0.100825,0.0354,...,0.0067,0.051323,0.0331,NaN,0.0331,NaN,0.786223,1965-01-01,0.0331,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,2020,0.037209,0.015096,3756.07,139.760000,56.700000,127.78,-0.139144,-0.035714,0.0009,...,0.0084,0.007352,0.0165,0.0542,0.0472,0.0494,5.075269,2021-01-01,0.0472,0.0542
61,2021,0.043301,0.012421,4766.18,206.380000,59.200000,147.24,0.476674,0.044092,0.0006,...,0.0145,0.017052,0.0172,0.0647,0.0424,0.0490,2.807947,2022-01-01,0.0424,0.0647
62,2022,0.057166,0.017799,3839.50,219.490000,68.340000,181.99,0.063524,0.154392,0.0442,...,-0.0054,0.055874,0.0216,0.0641,0.0594,0.0511,1.530928,2023-01-01,0.0594,0.0641
63,2023,0.046408,0.014690,4769.83,221.360000,70.070000,164.79,0.008520,0.025315,0.0520,...,-0.0132,0.036808,0.0197,0.0874,0.0460,0.0457,1.185567,2024-01-01,0.0460,0.0874


,Start of month,S&P 500,T.Bond Rate,Ten-year average CF,CF (Trailing 12 month),Normalized CF,Expected growth rate,ERP (T12 m with sustainable payout),ERP (T12m),ERP (Smoothed),ERP (Normalized),ERP (Net Cash Yield),ERP (Covid Adjusted),Expected Return,Notes,Date,rERP,rGROWTH
0,2008-09-01 00:00:00,1252.0,0.0372,NaN,NaN,NaN,NaN,NaN,0.0422,NaN,NaN,NaN,NaN,0.0794,NaN,2008-09-01,0.0422,NaN
1,2008-10-01 00:00:00,1166.0,0.0383,NaN,NaN,NaN,NaN,NaN,0.0451,NaN,NaN,NaN,NaN,0.0834,NaN,2008-10-01,0.0451,NaN
2,2008-11-01 00:00:00,969.0,0.0395,NaN,NaN,NaN,NaN,NaN,0.059,NaN,NaN,NaN,NaN,0.0985,NaN,2008-11-01,0.059,NaN
3,2008-12-01 00:00:00,896.0,0.0292,NaN,NaN,NaN,NaN,NaN,0.066,NaN,NaN,NaN,NaN,0.0952,NaN,2008-12-01,0.066,NaN
4,2009-01-01 00:00:00,903.0,0.0221,NaN,52.58,NaN,0.04,NaN,0.0643,NaN,NaN,NaN,NaN,0.0864,NaN,2009-01-01,0.0643,0.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2024-12-01 00:00:00,6032.0,0.0418,255.82,176.38,159.92,0.0955,0.0385,0.0407,0.0587,0.0369,0.0389,NaN,0.0825,Updated growth rates,2024-12-01,0.0407,0.0955
196,2025-01-01 00:00:00,5882.0,0.0458,261.11,182.79,160.5,0.0957,0.04,0.0433,0.0615,0.0381,0.0415,NaN,0.0891,"Updated cash flows, growth rates",2025-01-01,0.0433,0.0957
197,2025-02-01 00:00:00,6041.0,0.0454,261.11,182.79,160.5,0.0964,0.0395,0.0427,0.0606,0.0375,0.0406,NaN,0.0881,Updated growth rates,2025-02-01,0.0427,0.0964
198,2025-03-01 00:00:00,5955.0,0.0422,261.10,182.79,160.5,0.092,0.0412,0.0435,0.0618,0.0383,0.0409,NaN,0.0857,Updated growth rates,2025-03-01,0.0435,0.092


/var/folders/8g/0xm7kl8n0vv2_vb9k8nn22sh0000gn/T/ipykernel_16716/1469340414.py:26: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Date,rERP,T.Bond Rate,rGROWTH,S&P 500
0,2008-09-12,0.04222,0.0372,0.05,1252.00
1,2008-09-13,NaN,NaN,NaN,NaN
2,2008-09-14,NaN,NaN,NaN,NaN
3,2008-09-15,0.04480,0.0339,0.05,1193.00
4,2008-09-16,0.04390,0.0344,0.05,1214.00
...,...,...,...,...,...
6057,2025-04-13,NaN,NaN,NaN,NaN
6058,2025-04-14,0.04730,0.0438,NaN,5405.97
6059,2025-04-15,0.04740,0.0435,NaN,5396.63
6060,2025-04-16,0.04850,0.0429,NaN,5275.70


In [5]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=dfy["Date"], y=dfy["rERP"], mode="lines+markers", name="Yearly Estimate", 
                line=dict(color='limegreen', width=3)))
fig.add_trace(go.Scatter(x=dfm["Date"], y=dfm["rERP"], mode="lines", name="Monthly  Estimate",
                line=dict(color='darkgreen', width=2)))
fig.add_trace(go.Scatter(x=dfd["Date"], y=dfd["rERP"], mode="lines", name="Daily  Estimate",
                line=dict(color='blue', width=1)))
fig.update_layout(xaxis_title="Date of Estimation", yaxis_title="Implied ERP", yaxis_type="linear", yaxis_tickformat=',.0%')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dfy["Date"], y=dfy["T.Bond Rate"], mode="lines+markers", name="RFR Yearly Estimate",
              line=dict(color='violet', width=3)))
fig.add_trace(go.Scatter(x=dfm["Date"], y=dfm["T.Bond Rate"], mode="lines", name="RFR Monthly Estimate",
              line=dict(color='magenta', width=2)))
fig.add_trace(go.Scatter(x=dfd["Date"], y=dfd["T.Bond Rate"], mode="lines", name="RFR Daily Estimate",
              line=dict(color='black', width=1)))

fig.add_trace(go.Scatter(x=dfy["Date"], y=dfy["T.Bond Rate"]+dfy["rERP"], mode="lines+markers", name="RFR+ERP Yearly Estimate",
              line=dict(color='limegreen', width=3)))
fig.add_trace(go.Scatter(x=dfm["Date"], y=dfm["T.Bond Rate"]+dfm["rERP"], mode="lines", name="RFR+ERP  Monthly Estimate",
              line=dict(color='darkgreen', width=2)))
fig.add_trace(go.Scatter(x=dfd["Date"], y=dfd["T.Bond Rate"]+dfd["rERP"], mode="lines", name="RFR+ERP  Daily Estimate",
            line=dict(color='blue', width=1)))

fig.update_layout(xaxis_title="Date of Estimation", yaxis_title="T.Bond Rate + ERP", yaxis_type="linear", yaxis_tickformat=',.0%')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dfy["Date"], y=dfy["rGROWTH"], mode="lines+markers", name="Yearly Estimate",
            line=dict(color='limegreen', width=3)))
fig.add_trace(go.Scatter(x=dfm["Date"], y=dfm["rGROWTH"], mode="lines", name="Monthly  Estimate",
            line=dict(color='darkgreen', width=2)))
fig.update_layout(xaxis_title="Date of Estimation", yaxis_title="Growth estimate", yaxis_type="linear", yaxis_tickformat=',.0%')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dfy["Date"], y=dfy["S&P 500"], mode="lines+markers", name="Yearly Estimate",
            line=dict(color='limegreen', width=3)))
fig.add_trace(go.Scatter(x=dfm["Date"], y=dfm["S&P 500"], mode="lines", name="Monthly  Estimate",
            line=dict(color='darkgreen', width=2)))
fig.add_trace(go.Scatter(x=dfd["Date"], y=dfd["S&P 500"], mode="lines", name="Daily  Estimate",
            line=dict(color='blue', width=1)))
fig.update_layout(xaxis_title="Date of Estimation", yaxis_title="S&P 500 price", yaxis_type="log")
fig.show()

In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dfy["Analyst Growth Estimate"], y=dfy["Implied ERP (FCFE)"], 
    hovertext=[ f"Date: {x.date()}" for x in dfy["Date"]], 
    mode="markers", name="Yearly Estimate"))
fig.add_trace(go.Scatter(x=dfm["Expected growth rate"], y=dfm["ERP (T12m)"], 
    hovertext=[ f"Date: {x.date()}" for x in dfm["Date"]], 
    mode="markers", name="Monthly  Estimate"))
fig.add_trace(go.Scatter(x=dfm.iloc[[-1]]["Expected growth rate"], y=dfm.iloc[[-1]]["ERP (T12m)"], 
    hovertext=[ f"Date: {x.date()}" for x in dfm.iloc[[-1]]["Date"]],
    marker_size=20, marker_opacity=0.5, 
    mode="markers", name=f"Now - {dfm.iloc[-1]['Date'].date()}"))
fig.update_layout(
    margin=dict(l=10, r=10, t=10, b=10),
    xaxis_title="Growth Estimate", yaxis_title="Equity Risk Premium",
    xaxis_tickformat=',.0%', yaxis_tickformat=',.0%',
    width=1000, height=800)
fig.show()

## Conclusions
- Independently from the RFR (10-years US Treasury yield) which is a proxy for expected growth+inflation we can use the ERP (Equity Risk Premium) as a proxy for the market greed/fear
- When the ERP is high there is a lot of fear, e.g., the market expect oscillations in the near-future
- When the Growth Estimate is high, typically the market is overexcited, and the growth will mean-revert to some more reasonable long-term trends
- This last statement is more controversial because the analyst may already expect a low groewth after a bull run, still underestimating the downturn of the index: this is the case of low-GrowthEstimate in late 2019, when the analyst were conservative in projecting a lower growth after a booming year. Maybe not a good moment to enter the market, despite the already-low estimated growth expectation.
- As investors, if we assume that there won't be major market disruptions in the future, we want to enter the market when the ERP is high and the Growth Estimate is low (assuming analyst are already conservative), therefore we want to be in the top-left quadrant of the GrowthEst/ERP plot
- We are currently (May 2023) in a mildly good condition to enter the market, with below-average market growth expectation and quite average market fear/greed (considering the ERP). We are not in extreme condition to conclude the market is a bargain nor that it is overpriced.

## Follow-up
- Add the future 1-5-10 years growth, to check if the growth estimate was legit
- Include other macro indicators, to understand when the analyst are too optimistic/pessimistic in their Growth Estimate: this is not easy, as it is a recursive evaluation, i.e., using macro indicators to evaluate the analyst that are evaluating the macro indicators.